# Week 2: Advanced Pandas Data Wrangling
## Notebook 12: Vectorized String Operations & Regex Data Wrangling

### 1. Conceptual Intuition & Theoretical Background
Text features in raw production data (e.g., URLs, log files, addresses) are unstructured and noisy. Processing strings via standard Python list comprehensions creates significant interpreter overhead.

Pandas provides **Vectorized String Accessors (`.str`)**:
- Executes C-accelerated regex pattern matching across entire Series simultaneously.
- **Capture Groups:** Extracting multiple structured fields via named regular expression groups in a single pass.
- **Unicode Normalization (NFKD / NFC):** Decomposing composite Unicode characters (e.g. accents, emojis, non-breaking spaces) to ensure consistent text representations across international datasets.

---

### 2. Scratch Python Implementation: High-Throughput Regex Pipeline

In [ ]:
import pandas as pd
import unicodedata

# Raw noisy production user data
raw_logs = pd.Series([
    "ID: #US-8492 | User: Alice_Smith | IP: 192.168.1.10:8080 | Status: 200 OK | Action: /checkout?id=99",
    "ID: #EU-1204 | User: Björk_Guð | IP: 10.0.0.15:443 | Status: 404 NOT_FOUND | Action: /catalog",
    "ID: #AP-5531 | User: Chloë_Müller | IP: 172.16.254.1:80 | Status: 500 ERROR | Action: /api/v1/pay"
])

# 1. Regex Extraction using Named Capture Groups
regex_pattern = (
    r"ID:s*#(?P<Region>[A-Z]{2})-(?P<ID>d+)s*|s*"
    r"User:s*(?P<Username>[^|]+)s*|s*"
    r"IP:s*(?P<IP_Address>[d.]+):(?P<Port>d+)s*|s*"
    r"Status:s*(?P<Status_Code>d+)"
)

extracted_df = raw_logs.str.extract(regex_pattern)

# 2. Unicode Normalization (decompose accents e.g., 'ö' -> 'o' + umlaut)
def clean_unicode(text):
    if not isinstance(text, str):
        return text
    # Normalize to NFKD form and strip non-ASCII combining marks
    normalized = unicodedata.normalize('NFKD', text.strip())
    return "".join(c for c in normalized if not unicodedata.combining(c))

extracted_df['Clean_Username'] = extracted_df['Username'].apply(clean_unicode)
extracted_df['Status_Code'] = extracted_df['Status_Code'].astype(int)

print("--- Parsed Structured Log Data ---")
print(extracted_df[['Region', 'ID', 'Clean_Username', 'IP_Address', 'Port', 'Status_Code']])

### 3. Step-by-Step Code Walkthrough
1. **Named Groups (`(?P<name>...)`)**: Automatically names DataFrame columns based on regex capture groups.
2. **NFKD Normalization**: Separates accented characters from base letters, enabling uniform cleaning of international text.
3. **Type Coercion**: Casts parsed string tokens into numeric dtypes.

---

### 4. Industry Applications & Trade-offs
- **Cybersecurity & SIEM Telemetry:** Parsing streaming syslog files and firewall logs into structured feature tables.
- **Trade-off:** Complex regex with backtracking can cause catastrophic performance slowdowns on malformed strings.

---

### 5. Top Interview Questions & Answers
**Q1: What is the performance difference between `df['col'].apply(lambda x: re.match(...))` and `df['col'].str.extract(...)`?**
> **Answer:** `.str.extract` compiles the regex pattern once and iterates through C-level buffers, while Python `.apply()` incurs function invocation overhead for every single row.

**Q2: Why is Unicode normalization necessary prior to NLP tokenization?**
> **Answer:** Characters like "é" can be stored as either a single code point (`\u00E9`) or two code points (`e` + combining accent `\u0301`). Normalization ensures consistent vocabulary IDs.